# Panel construction

Two steps. First the crisis label is built from Laeven and Valencia (2026), then the micro
panel, the macro panel and that label are merged into the modelling datasets.

**Outputs** `data/merged/parquet_saves/merged_base.parquet` and `merged_robust.parquet`,
plus the unlabelled versions.

Merged from the original `08_Crisis_dataset` and `09_mergedsets`.

## Setup

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pycountry

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

CRISIS_RAW = Path("../data/crisis/raw/SYST_BANK_CRIS_updated.xlsx")
CRISIS_CLEAN = Path("../data/crisis/clean")
MACRO_CLEAN = Path("../data/macro/clean/parquet_saves")
MICRO_CLEAN = Path("../data/micro/clean")
MERGED_DIR = Path("../data/merged")

for d in (CRISIS_CLEAN, CRISIS_CLEAN / "parquet_saves", MERGED_DIR, MERGED_DIR / "parquet_saves"):
    d.mkdir(parents=True, exist_ok=True)

# Part 1. Crisis panel

# Crisis duration panel (Laeven & Valencia 2026)

Previous version used onset-only labeling: `crisis=1` only in the start year.
This placed all positives in 2008 (the training window), leaving every test year (2010-2017) with zero positives - model evaluation was impossible.

This version uses **duration labeling**: `crisis=1` for every year from onset through the end year.

End dates: Laeven & Valencia (2026) provide explicit end years for all episodes. A five-year truncation rule is applied: where distinguishing a late-stage crisis from a weak economy was not possible, episodes are coded as ending no later than five years after onset. GFC-era crises therefore end in 2009–2012 depending on onset year.


**Pre-1995 crises:** Crises that started before our sample window are still expanded. Years before 1995
are dropped automatically when we left-join onto the skeleton (which starts at 1995). For example,
Finland 1991-1995 contributes only the 1995 observation to the robust panel.

In [ ]:
res = pd.read_excel(CRISIS_RAW, sheet_name="Sheet1") #the Crisis Resolution and Outcomes sheet, which carries both Start and End years. The Crisis Years sheet only has start years, which is what the earlier onset-only version used
print("Loaded shape:", res.shape)
print(res[["Country", "Start", "End"]].head(10).to_string())

## Clean country names

In [ ]:
# L&V appends footnote numbers to country names, e.g. "Albania 13/" -> "Albania"
# Pattern: one or more whitespace, one or more digits, a slash, and anything after
res["Country_clean"] = (
    res["Country"]
    .astype(str)
    .str.replace(r"\s+\d+/.*", "", regex=True)
    .str.strip()
)

## Filter to the EU-25 sample

In [ ]:
# Cyprus and Malta excluded from our sample due to data availability
# Slovak Republic appears as "Slovak Rep." in the L&V Resolution sheet
EU25_NAMES = {
    "Austria", "Belgium", "Bulgaria", "Croatia", "Czech Republic",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg",
    "Netherlands", "Poland", "Portugal", "Romania", "Slovak Republic",
    "Slovenia", "Spain", "Sweden",
}

res_eu = res[res["Country_clean"].isin(EU25_NAMES)][["Country_clean", "Start", "End"]].copy()
print(res_eu.to_string())

# Flag countries expected in our sample but missing from the Resolution sheet
# Missing means no systemic banking crisis per L&V 2026, or crisis fully predates the sample
missing = EU25_NAMES - set(res["Country_clean"].unique())
if missing:
    print(f"\nNot found in Resolution sheet: {missing}")
else:
    print("\nAll EU-25 names found in Resolution sheet.")

## Parse Start and End years

L&V year values come in up to three forms:
1. A plain year: `1997`
2. A year with a footnote suffix: `"2000 11/"` -> strip suffix, keep year

One helper function handles all three cases; `ongoing_year=None` disables the ongoing check for Start.

In [ ]:
def parse_year(val, ongoing_year=None):
    """
    Parses a year value from L&V.
    If ongoing_year is set, 'ongoing' maps to that value (used for End column).
    Footnote suffixes like '11/' are stripped before parsing.
    Returns None if parsing fails.
    """
    s = str(val).strip()
    if ongoing_year is not None and s.lower() == "ongoing":
        return ongoing_year
    s = re.sub(r"\s+\d+/.*", "", s)  # strip trailing footnote reference, e.g. " 11/"
    try:
        return int(float(s))
    except (ValueError, TypeError):
        return None

res_eu["start_year"] = res_eu["Start"].apply(lambda x: parse_year(x))
res_eu["end_year"]   = res_eu["End"].apply(lambda x: parse_year(x))

n_before = len(res_eu)
res_eu = res_eu.dropna(subset=["start_year", "end_year"])
res_eu["start_year"] = res_eu["start_year"].astype(int)
res_eu["end_year"]   = res_eu["end_year"].astype(int)

if len(res_eu) < n_before:
    print(f"Dropped {n_before - len(res_eu)} rows with unparseable dates.")

print(res_eu[["Country_clean", "start_year", "end_year"]].to_string())

## Expand each crisis episode into one row per crisis year

Each row in the Resolution sheet is one crisis episode for one country.
We expand start -> end into individual year observations so we can merge with the panel skeleton.

Years before 1995 (from pre-sample crises) will be dropped silently by the left join onto the skeleton.

In [ ]:
rows = []
for _, row in res_eu.iterrows():
    for yr in range(row["start_year"], row["end_year"] + 1):
        rows.append({"Country_clean": row["Country_clean"], "year": yr})

crisis_years = pd.DataFrame(rows)
print("Total crisis country-year rows before dedup:", len(crisis_years))
print("\nAustria as example (should span 2008-2012 with L&V 2026 5 year truncation):")
print(crisis_years[crisis_years["Country_clean"] == "Austria"].to_string())

## Map country names to ISO3

In [ ]:
# pycountry does not recognise all L&V name variants - override manually where needed
OVERRIDES = {
    "Slovak Republic": "SVK",   # L&V Resolution sheet abbreviation for Slovak Republic
    "Czech Republic": "CZE",
}

def name_to_iso3(name):
    if name in OVERRIDES:
        return OVERRIDES[name]
    try:
        return pycountry.countries.get(name=name).alpha_3
    except Exception:
        return None

crisis_years["country_iso3"] = crisis_years["Country_clean"].apply(name_to_iso3)

# Any NaN here means an unhandled name variant - add it to OVERRIDES above
unmatched = crisis_years[crisis_years["country_iso3"].isna()]["Country_clean"].unique()
if len(unmatched):
    print("WARNING - unmapped countries (add to OVERRIDES):", unmatched)
else:
    print("All countries mapped to ISO3.")

## Deduplicate

Some countries have two separate crisis episodes in L&V (e.g. Latvia: 1995-1996 and 2008-ongoing;
Spain: 1977-1981 and 2008-ongoing). Deduplication ensures each country-year appears once —
a country in two overlapping crises is still just crisis=1.

In [ ]:
crisis_years_dedup = (
    crisis_years[["country_iso3", "year"]]
    .drop_duplicates()
    .sort_values(["country_iso3", "year"])
    .reset_index(drop=True)
)
print("Unique crisis country-years before skeleton filter:", len(crisis_years_dedup))
print("\nCrisis count by year (all years, including pre-1995):")
print(crisis_years_dedup.groupby("year").size().to_string())

# Create skeleton

25 countries x 1995-2017 = 575 rows.
Country list drawn from macro_robust_1995_2017 to ensure exact match with the macro panel.

In [ ]:
base = pd.read_parquet(MACRO_CLEAN / "macro_robust_1995_2017.parquet")
print(base["country_iso3"].nunique(), "unique countries in macro panel") #taken from the macro panel rather than hard-coded, so the two can never drift apart

idx = pd.MultiIndex.from_product( #from_product creates every possible country-year combination, so the skeleton is complete by construction
    [base["country_iso3"].unique(), range(1995, 2018)],
    names=["country_iso3", "year"],
)
skeleton = idx.to_frame(index=False)
print("Skeleton shape:", skeleton.shape)

skeleton.to_parquet(CRISIS_CLEAN / "parquet_saves" / "Skeleton.parquet")

# Create crisis panel

Left join the skeleton onto crisis_years_dedup on (country_iso3, year).
Matching rows get crisis=1; unmatched rows get crisis=0.
Crisis years outside the skeleton range (pre-1995) are dropped automatically by the left join.

In [ ]:
# assign(crisis=1) adds the crisis column to the dedup table before merging
# Left join keeps all 575 skeleton rows; NaN where no crisis match -> filled with 0
crisis_panel = skeleton.merge(
    crisis_years_dedup.assign(crisis=1),
    on=["country_iso3", "year"],
    how="left"
)
crisis_panel["crisis"] = crisis_panel["crisis"].fillna(0).astype(int)
print("Shape:", crisis_panel.shape)

## Checks

In [ ]:
print("Crisis value counts:")
print(crisis_panel["crisis"].value_counts())

print("\nCrisis=1 observations by year:")
print(crisis_panel.groupby("year")["crisis"].sum().to_string())

print("\nCrisis=1 total per country:")
print(crisis_panel.groupby("country_iso3")["crisis"].sum().sort_values(ascending=False).to_string())

## Save

In [ ]:
crisis_panel.to_parquet(CRISIS_CLEAN / "parquet_saves" / "crisis_panel.parquet")
print("Saved crisis_panel:", crisis_panel.shape)

# Part 2. Merge micro, macro and crisis

The same three steps run twice, once for the baseline set and once for the robustness set.
Each macro country-year row is attached to every bank in that country, which is the
duplication the thesis discusses as the reason integration does not compound.

## Baseline

In [ ]:
micro_set_base = pd.read_parquet(MICRO_CLEAN / "micro_clean_base.parquet")
macro_set_base = pd.read_parquet(MACRO_CLEAN / "macro_combi_base_2004_2017.parquet")
crisis_panel_base = pd.read_parquet(CRISIS_CLEAN / "parquet_saves" / "crisis_panel.parquet")

for name, df in [("micro_set_base", micro_set_base),
                 ("macro_set_base", macro_set_base),
                 ("crisis_panel_base", crisis_panel_base)]:
    print(f"{name}: {df['year'].min()} - {df['year'].max()} ({len(df)} rows)")

In [ ]:
base_merged = micro_set_base.merge( #left join on the micro panel, so every bank-year is kept and the country's macro row is attached to each of its banks
    macro_set_base, left_on=["country_iso", "year"],
    right_on=["country_iso3", "year"], how="left",
)
base_merged = base_merged.drop(columns=["country_iso3"]) #dropped now so the crisis merge below doesn't produce _x/_y duplicates

base_merged.to_parquet(MERGED_DIR / "parquet_saves" / "merged_base_unlabelled.parquet")
print("Saved merged_base_unlabelled:", base_merged.shape)

In [ ]:
base_labelled = base_merged.merge(
    crisis_panel_base, left_on=["country_iso", "year"],
    right_on=["country_iso3", "year"], how="left",
)
base_labelled = base_labelled.drop(columns=["country_iso3"])
base_labelled["crisis"] = base_labelled["crisis"].fillna(0).astype(int) #a bank-year with no matching country-year in the crisis panel is not in a crisis

print("Labelled base:", base_labelled.shape)

### Checks

In [ ]:
def check_merge(micro, merged, labelled, name):
    print(f"=== {name} ===")

    # 1) Row count preserved, a left join should never add or drop rows
    print("Rows:", len(micro), "->", len(merged), "->", len(labelled))

    # 2) No unmatched macro rows, a NaN here means a micro country-year had no macro match
    macro_cols = [c for c in ["NGDP_RPCH", "PCPIPCH", "LUR"] if c in merged.columns]
    unmatched = merged[merged[macro_cols[0]].isna()][["bank_id", "country_iso", "year"]]
    print("Unmatched macro rows:", len(unmatched))
    if len(unmatched):
        print(unmatched["country_iso"].value_counts().to_string())

    # 3) Crisis distribution
    counts = labelled["crisis"].value_counts()
    print("Crisis rows:", counts.to_dict(), f"({labelled['crisis'].mean():.1%} positive)")

    # 4) Which country-years carry crisis=1
    cy = labelled.loc[labelled["crisis"] == 1, ["country_iso", "year"]].drop_duplicates()
    print("Crisis country-years:", len(cy))

    # 5) No country_iso3 column left behind
    print("Lingering iso3 columns:", [c for c in labelled.columns if "iso3" in c])


check_merge(micro_set_base, base_merged, base_labelled, "baseline")

The crisis distribution is printed by the check above. Crisis years are the minority class but not a
rare one (about a fifth of bank-years in the lagged modelling panels), because many EU countries spent
several years of 2008-2012 in crisis. `class_weight='balanced'` and `n_neg_ratio` in the modelling
notebooks reweight the classes per training window. (The earlier note here quoted an older labelling,
3,164 crisis bank-years, 68%, which no longer applies: audit A31.)

Marks all years of a crisis duration as 1, not just onset year. The model predicts "is this country in a crisis year" not "is a crisis about to start." With t-1 lagged features, the row for year t uses features from t-1, so for the first crisis year the model sees pre-crisis features → crisis label. For continuation years (t+1, t+2...) the model sees mid-crisis features → still crisis label. This is why RT1 (removing onset+t+1 from training) is important as a robustness check — it tests whether your model can learn pre-crisis signals OR just recognizes what mid-crisis financial state looks like.

### Save

In [ ]:
base_labelled.to_parquet(MERGED_DIR / "parquet_saves" / "merged_base.parquet")
print("Saved merged_base:", base_labelled.shape)

## Robustness

In [ ]:
micro_set_robust = pd.read_parquet(MICRO_CLEAN / "micro_clean_robust.parquet")
macro_set_robust = pd.read_parquet(MACRO_CLEAN / "macro_combi_robust_2004_2017.parquet")
crisis_panel_robust = pd.read_parquet(CRISIS_CLEAN / "parquet_saves" / "crisis_panel.parquet")

for name, df in [("micro_set_robust", micro_set_robust),
                 ("macro_set_robust", macro_set_robust),
                 ("crisis_panel_robust", crisis_panel_robust)]:
    print(f"{name}: {df['year'].min()} - {df['year'].max()} ({len(df)} rows)")

micro_set_robust_filtered = micro_set_robust[micro_set_robust["year"] >= 2004].copy() #the robust micro panel runs from 1995, but the macro panel it merges onto starts in 2004, so the extra years would only produce unmatched rows
print("Micro robust:", len(micro_set_robust), "->", len(micro_set_robust_filtered))

In [ ]:
robust_merged = micro_set_robust_filtered.merge(
    macro_set_robust, left_on=["country_iso", "year"],
    right_on=["country_iso3", "year"], how="left",
)
robust_merged = robust_merged.drop(columns=["country_iso3"])

robust_merged.to_parquet(MERGED_DIR / "parquet_saves" / "merged_robust_unlabelled.parquet")
print("Saved merged_robust_unlabelled:", robust_merged.shape)

In [ ]:
robust_labelled = robust_merged.merge(
    crisis_panel_robust, left_on=["country_iso", "year"],
    right_on=["country_iso3", "year"], how="left",
)
robust_labelled = robust_labelled.drop(columns=["country_iso3"])
robust_labelled["crisis"] = robust_labelled["crisis"].fillna(0).astype(int)

check_merge(micro_set_robust_filtered, robust_merged, robust_labelled, "robustness")

### Save

In [ ]:
robust_labelled.to_parquet(MERGED_DIR / "parquet_saves" / "merged_robust.parquet")
print("Saved merged_robust:", robust_labelled.shape)